In [ ]:
import os
import glob
from dotenv import load_dotenv

load_dotenv()

from langchain_openai import OpenAIEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS


# ----------------------------
# CONFIG
# ----------------------------
PDF_FOLDER =  r"C:\Users\surya.adatravu\Documents\ContextRAG\pdfs_folder"
INDEX_DIR = r"C:\Users\surya.adatravu\Documents\ContextRAG\vector_db"


# ----------------------------
# EMBEDDINGS
# ----------------------------
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")


# ----------------------------
# LOAD PDFS
# ----------------------------
def load_pdfs_from_folder(folder_path: str):
    pdf_paths = sorted(glob.glob(os.path.join(folder_path, "*.pdf")))
    if not pdf_paths:
        raise FileNotFoundError(f"No PDF files found in: {folder_path}")

    docs = []
    for p in pdf_paths:
        loader = PyPDFLoader(p)
        pages = loader.load()  # one Document per page
        for d in pages:
            d.metadata["source_file"] = os.path.basename(p)
        docs.extend(pages)

    return docs


# ----------------------------
# CHUNK
# ----------------------------
def chunk_documents(docs, chunk_size=900, chunk_overlap=150):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=["\n\n", "\n", " ", ""],
    )
    chunks = splitter.split_documents(docs)
    for i, c in enumerate(chunks):
        c.metadata["chunk_id"] = i
    return chunks


# ----------------------------
# BUILD + SAVE FAISS
# ----------------------------
def build_and_save_faiss(chunks, index_dir: str):
    os.makedirs(index_dir, exist_ok=True)
    vs = FAISS.from_documents(chunks, embeddings)
    vs.save_local(index_dir)
    return vs


if __name__ == "__main__":
    docs = load_pdfs_from_folder(PDF_FOLDER)
    chunks = chunk_documents(docs)

    build_and_save_faiss(chunks, INDEX_DIR)

    print(f"✅ Built FAISS index from PDFs.")
    print(f"   PDF folder: {PDF_FOLDER}")
    print(f"   Index dir : {INDEX_DIR}")
    print(f"   Pages     : {len(docs)}")
    print(f"   Chunks    : {len(chunks)}")
